# Recomendation System: Evaluación de Fairness

En esta notebook realizamos un benchmarking de un algoritmo SVD utilizando la librería Surprise sobre el dataset Book-Crossing. Nos centraremos en segmentar a los usuarios por grupos demográficos (Edad, Geografía) y de comportamiento (Mainstream vs. Nicho) para evaluar posibles sesgos mediante métricas de ranking realistas y pruebas de significancia estadística.

### 1. Importación y Carga de Datos

Importamos las librerías necesarias y cargamos los conjuntos de datos de interacciones y usuarios.

In [1]:
import pandas as pd
import numpy as np
import random
from collections import defaultdict
import scipy.stats as stats
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

ratings_path = "./datasets/BX-Book-Ratings.csv"
users_path = "./datasets/BX-Users.csv"

ratings = pd.read_csv(ratings_path, sep=';', encoding='latin-1', on_bad_lines='skip')
users = pd.read_csv(users_path, sep=';', encoding='latin-1', on_bad_lines='skip')

ratings.columns = ratings.columns.str.strip()
users.columns = users.columns.str.strip()

### 2. Segmentación Demográfica

Filtramos las edades válidas y creamos variables categóricas para agrupar por rango etario y región geográfica.

In [2]:
users_v = users[(users['Age'] >= 10) & (users['Age'] <= 90)].copy()
users_v['Age_Group'] = users_v['Age'].apply(lambda x: 'Joven' if x <= 35 else 'Adulto')
users_v['Country'] = users_v['Location'].apply(lambda x: str(x).split(',')[-1].strip().lower())
users_v['Is_USA_or_Can'] = users_v['Country'].isin(['usa', 'canada'])

print("Distribución por Edad:\n", users_v['Age_Group'].value_counts())
print("\nDistribución por Geografía (USA/Can vs Resto):\n", users_v['Is_USA_or_Can'].value_counts())

Distribución por Edad:
 Age_Group
Joven     98206
Adulto    68391
Name: count, dtype: int64

Distribución por Geografía (USA/Can vs Resto):
 Is_USA_or_Can
True     87224
False    79373
Name: count, dtype: int64


### 3. Filtrado de Interacciones y Configuración del Modelo

Nos quedamos con calificaciones explícitas de usuarios que posean un mínimo de interacciones y realizamos la partición en entrenamiento y prueba.

In [3]:
df_completo = pd.merge(ratings, users_v, on='User-ID', how='inner')
df_completo = df_completo[df_completo['Book-Rating'] > 0]
user_counts = df_completo['User-ID'].value_counts()
df_completo = df_completo[df_completo['User-ID'].isin(user_counts[user_counts >= 8].index)]

df_surprise = df_completo[['User-ID', 'ISBN', 'Book-Rating']].copy()
df_surprise.columns = ['userID', 'itemID', 'rating']
reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(df_surprise, reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

### 4. Segmentación por Comportamiento

Dividimos a los usuarios en **Mainstream** y **Nicho** calculando la popularidad de los ítems usando exclusivamente el conjunto de entrenamiento.

In [4]:
train_rows = [(trainset.to_raw_uid(uid), trainset.to_raw_iid(iid), r) 
              for uid, ir in trainset.ur.items() for iid, r in ir]
df_train = pd.DataFrame(train_rows, columns=['User-ID', 'ISBN', 'Book-Rating'])

libro_counts_train = df_train['ISBN'].value_counts()
umbral_popularidad_train = libro_counts_train.quantile(0.80)

ratings_pop_train = df_train.merge(libro_counts_train.rename('Votos_Libro'), left_on='ISBN', right_index=True)
ratings_pop_train['es_popular'] = (ratings_pop_train['Votos_Libro'] >= umbral_popularidad_train).astype(int)

user_ratio_popular_train = ratings_pop_train.groupby('User-ID')['es_popular'].mean()
usuarios_mainstream = user_ratio_popular_train[user_ratio_popular_train >= 0.80].index
usuarios_nicho = user_ratio_popular_train[user_ratio_popular_train < 0.20].index

print(f"Umbral de popularidad en train: {umbral_popularidad_train} votos")
print(f"Usuarios Mainstream: {len(usuarios_mainstream)} | Usuarios Nicho: {len(usuarios_nicho)}")

Umbral de popularidad en train: 2.0 votos
Usuarios Mainstream: 1776 | Usuarios Nicho: 447


### 5. Entrenamiento del Modelo Predictivo

Ajustamos el algoritmo SVD, generamos predicciones y consolidamos un dataset de errores.

In [5]:
algo = SVD(random_state=42)
algo.fit(trainset)
predictions = algo.test(testset)

df_preds = pd.DataFrame(predictions, columns=['uid', 'iid', 'r_ui', 'est', 'details'])
df_preds = df_preds.merge(users_v[['User-ID', 'Age', 'Is_USA_or_Can']], left_on='uid', right_on='User-ID', how='inner')
df_preds['Es_Mainstream'] = df_preds['uid'].isin(usuarios_mainstream)
df_preds['Es_Nicho'] = df_preds['uid'].isin(usuarios_nicho)
df_preds['Error_Absoluto'] = np.abs(df_preds['r_ui'] - df_preds['est'])

### 6. Evaluación Predictiva (RMSE y MAE)

Comparamos la desviación predictiva para evaluar sesgos utilizando RMSE y el test estadístico U de Mann-Whitney.

In [6]:
grupos = {
    'Jóvenes': df_preds['Age'] <= 35, 'Adultos': df_preds['Age'] > 35,
    'USA/Can': df_preds['Is_USA_or_Can'], 'Resto Mundo': ~df_preds['Is_USA_or_Can'],
    'Mainstream': df_preds['Es_Mainstream'], 'Nicho': df_preds['Es_Nicho']
}

print("RMSE POR GRUPOS")
for nombre, mascara in grupos.items():
    rmse_val = np.sqrt((df_preds.loc[mascara, 'Error_Absoluto']**2).mean())
    print(f"RMSE {nombre:12}: {rmse_val:.4f}")

user_errors = df_preds.groupby('uid').first()[['Age', 'Is_USA_or_Can', 'Es_Mainstream', 'Es_Nicho']]
user_errors['MAE'] = df_preds.groupby('uid')['Error_Absoluto'].mean()

print("\nP-VALUES (TEST MANN-WHITNEY SOBRE MAE POR USUARIO)")
_, p_edad = stats.mannwhitneyu(user_errors[user_errors['Age'] <= 35]['MAE'], user_errors[user_errors['Age'] > 35]['MAE'])
_, p_geo = stats.mannwhitneyu(user_errors[user_errors['Is_USA_or_Can']]['MAE'], user_errors[~user_errors['Is_USA_or_Can']]['MAE'])
_, p_gustos = stats.mannwhitneyu(user_errors[user_errors['Es_Mainstream']]['MAE'], user_errors[user_errors['Es_Nicho']]['MAE'])

print(f"Edad p-value: {p_edad:.4e} | Geografía p-value: {p_geo:.4e} | Gustos p-value: {p_gustos:.4e}")

RMSE POR GRUPOS
RMSE Jóvenes     : 1.5797
RMSE Adultos     : 1.4958
RMSE USA/Can     : 1.5372
RMSE Resto Mundo : 1.5495
RMSE Mainstream  : 1.5708
RMSE Nicho       : 1.5100

P-VALUES (TEST MANN-WHITNEY SOBRE MAE POR USUARIO)
Edad p-value: 3.9382e-01 | Geografía p-value: 2.9906e-01 | Gustos p-value: 2.2921e-01


### 7. Preparación para Métricas de Ranking

Muestreamos ítems negativos por usuario simulando un catálogo realista para calcular métricas Top-N.

In [7]:
random.seed(42)
top_n = defaultdict(list)
all_items = set(trainset.all_items())

for uid, iid, true_r, est, _ in predictions:
    top_n[uid].append((iid, est, true_r))

for uid_inner in trainset.all_users():
    uid_raw = trainset.to_raw_uid(uid_inner)
    if uid_raw not in top_n: continue
    
    test_items_inner = set()
    for iid_test, _, _ in top_n[uid_raw]:
        try:
            test_items_inner.add(trainset.to_inner_iid(iid_test))
        except ValueError:
            pass

    user_items_inner = set(j for (j, _) in trainset.ur[uid_inner]) | test_items_inner
    unrated_inner = list(all_items - user_items_inner)
    sampled_unrated = random.sample(unrated_inner, min(100, len(unrated_inner)))
    
    for iid_inner in sampled_unrated:
        iid_raw = trainset.to_raw_iid(iid_inner)
        est = algo.predict(uid_raw, iid_raw).est
        top_n[uid_raw].append((iid_raw, est, 0.0))

### 8. Cálculo de Precision, Recall y Cobertura

Calculamos métricas estándar de ranking para evaluar la calidad del sistema recomendador dentro de los primeros 5 resultados.

In [8]:
k = 5
threshold = 7
precisions, recalls = {}, {}
items_recomendados_por_grupo = defaultdict(set)
total_items_catalog = df_completo['ISBN'].nunique()

user_info = user_errors.to_dict('index')

for uid, user_ratings in top_n.items():
    user_ratings.sort(key=lambda x: x[1], reverse=True)
    top_k = user_ratings[:k]
    
    n_rel = sum(r_ui >= threshold for _, _, r_ui in user_ratings)
    n_rel_and_rec_k = sum(r_ui >= threshold for _, _, r_ui in top_k)
    
    precisions[uid] = n_rel_and_rec_k / k
    recalls[uid] = n_rel_and_rec_k / n_rel if n_rel else 0.0
    
    info = user_info.get(uid, {})
    grupos_usuario = []
    if info.get('Age', 99) <= 35: grupos_usuario.append('Jóvenes')
    else: grupos_usuario.append('Adultos')
    if info.get('Is_USA_or_Can', False): grupos_usuario.append('USA/Can')
    else: grupos_usuario.append('Resto Mundo')
    if info.get('Es_Mainstream', False): grupos_usuario.append('Mainstream')
    if info.get('Es_Nicho', False): grupos_usuario.append('Nicho')
        
    for g in grupos_usuario:
        items_recomendados_por_grupo[g].update([iid for iid, _, _ in top_k])

df_metrics = pd.DataFrame([{'uid': u, 'P5': p, 'R5': recalls[u]} for u, p in precisions.items()])
df_metrics = df_metrics.merge(user_errors.reset_index(), on='uid')

grupos_eval = {
    'Jóvenes': df_metrics['Age'] <= 35, 'Adultos': df_metrics['Age'] > 35,
    'USA/Can': df_metrics['Is_USA_or_Can'], 'Resto Mundo': ~df_metrics['Is_USA_or_Can'],
    'Mainstream': df_metrics['Es_Mainstream'], 'Nicho': df_metrics['Es_Nicho']
}

print("PRECISION@5 Y RECALL@5")
for n, m in grupos_eval.items():
    if m.sum() > 0:
        print(f"{n:12} -> P@5: {df_metrics.loc[m, 'P5'].mean():.4f} | R@5: {df_metrics.loc[m, 'R5'].mean():.4f}")

print("\nCOBERTURA (TOP-5)")
for n in grupos_eval.keys():
    cobertura = (len(items_recomendados_por_grupo[n]) / total_items_catalog) * 100
    print(f"{n:12} -> {cobertura:.2f}%")

PRECISION@5 Y RECALL@5
Jóvenes      -> P@5: 0.1088 | R@5: 0.1190
Adultos      -> P@5: 0.1001 | R@5: 0.1038
USA/Can      -> P@5: 0.1203 | R@5: 0.1288
Resto Mundo  -> P@5: 0.0691 | R@5: 0.0743
Mainstream   -> P@5: 0.1243 | R@5: 0.1776
Nicho        -> P@5: 0.0252 | R@5: 0.0352

COBERTURA (TOP-5)
Jóvenes      -> 9.11%
Adultos      -> 7.45%
USA/Can      -> 10.31%
Resto Mundo  -> 5.82%
Mainstream   -> 5.05%
Nicho        -> 1.62%


### 9. Test de Significancia para Ranking

Aplicamos el test de Mann-Whitney para determinar si las variaciones de rendimiento en Precision y Recall son estadísticamente significativas.

In [9]:
print("P-VALUES (TEST MANN-WHITNEY SOBRE P@5 Y R@5)")
_, p_edad_p5 = stats.mannwhitneyu(df_metrics.loc[df_metrics['Age'] <= 35, 'P5'], df_metrics.loc[df_metrics['Age'] > 35, 'P5'])
_, p_geo_p5 = stats.mannwhitneyu(df_metrics.loc[df_metrics['Is_USA_or_Can'], 'P5'], df_metrics.loc[~df_metrics['Is_USA_or_Can'], 'P5'])
_, p_gustos_p5 = stats.mannwhitneyu(df_metrics.loc[df_metrics['Es_Mainstream'], 'P5'], df_metrics.loc[df_metrics['Es_Nicho'], 'P5'])
print(f"P@5 -> Edad p-value: {p_edad_p5:.4e} | Geografía p-value: {p_geo_p5:.4e} | Gustos p-value: {p_gustos_p5:.4e}")

_, p_edad_r5 = stats.mannwhitneyu(df_metrics.loc[df_metrics['Age'] <= 35, 'R5'], df_metrics.loc[df_metrics['Age'] > 35, 'R5'])
_, p_geo_r5 = stats.mannwhitneyu(df_metrics.loc[df_metrics['Is_USA_or_Can'], 'R5'], df_metrics.loc[~df_metrics['Is_USA_or_Can'], 'R5'])
_, p_gustos_r5 = stats.mannwhitneyu(df_metrics.loc[df_metrics['Es_Mainstream'], 'R5'], df_metrics.loc[df_metrics['Es_Nicho'], 'R5'])
print(f"R@5 -> Edad p-value: {p_edad_r5:.4e} | Geografía p-value: {p_geo_r5:.4e} | Gustos p-value: {p_gustos_r5:.4e}")

P-VALUES (TEST MANN-WHITNEY SOBRE P@5 Y R@5)
P@5 -> Edad p-value: 9.3134e-03 | Geografía p-value: 5.6950e-31 | Gustos p-value: 1.2414e-34
R@5 -> Edad p-value: 1.8846e-03 | Geografía p-value: 3.7779e-28 | Gustos p-value: 1.0643e-34


### 10. Experimento de Impacto 50/50: Entrenamiento

Limitamos los datos de entrenamiento a la mitad para observar si el modelo penaliza a algún grupo acentuando las brechas de Fairness.

In [10]:
trainset_50, testset_50 = train_test_split(data, test_size=0.5, random_state=42)
algo_50 = SVD(random_state=42)
algo_50.fit(trainset_50)
predictions_50 = algo_50.test(testset_50)

df_train_50 = pd.DataFrame([(trainset_50.to_raw_uid(uid), trainset_50.to_raw_iid(iid), r) 
                            for uid, ir in trainset_50.ur.items() for iid, r in ir], 
                           columns=['User-ID', 'ISBN', 'Book-Rating'])

libro_counts_50 = df_train_50['ISBN'].value_counts()
umbral_pop_50 = libro_counts_50.quantile(0.80)
rat_pop_50 = df_train_50.merge(libro_counts_50.rename('Votos_Libro'), left_on='ISBN', right_index=True)
rat_pop_50['es_popular'] = (rat_pop_50['Votos_Libro'] >= umbral_pop_50).astype(int)
user_ratio_50 = rat_pop_50.groupby('User-ID')['es_popular'].mean()

usuarios_mainstream_50 = user_ratio_50[user_ratio_50 >= 0.80].index
usuarios_nicho_50 = user_ratio_50[user_ratio_50 < 0.20].index

### 11. Experimento de Impacto 50/50: Evaluación de Deltas

Calculamos la nueva disparidad en métricas de error tras reducir el conjunto de entrenamiento.

In [11]:
df_preds_50 = pd.DataFrame(predictions_50, columns=['uid', 'iid', 'r_ui', 'est', 'details'])
df_preds_50 = df_preds_50.merge(users_v[['User-ID', 'Age', 'Is_USA_or_Can']], left_on='uid', right_on='User-ID', how='inner')
df_preds_50['Error_Absoluto'] = np.abs(df_preds_50['r_ui'] - df_preds_50['est'])

grupos_50 = {
    'Jóvenes': df_preds_50['Age'] <= 35, 'Adultos': df_preds_50['Age'] > 35,
    'USA/Can': df_preds_50['Is_USA_or_Can'], 'Resto Mundo': ~df_preds_50['Is_USA_or_Can'],
    'Mainstream': df_preds_50['uid'].isin(usuarios_mainstream_50), 
    'Nicho': df_preds_50['uid'].isin(usuarios_nicho_50)
}

print("EXPERIMENTO 50/50: DELTAS ABSOLUTOS DE RMSE")
rmse_80, rmse_50 = {}, {}
for n, m in grupos.items(): rmse_80[n] = np.sqrt((df_preds.loc[m, 'Error_Absoluto']**2).mean())
for n, m in grupos_50.items(): rmse_50[n] = np.sqrt((df_preds_50.loc[m, 'Error_Absoluto']**2).mean())

print(f"Delta Edad (Original 80/20): {abs(rmse_80['Jóvenes'] - rmse_80['Adultos']):.4f} | Nuevo (50/50): {abs(rmse_50['Jóvenes'] - rmse_50['Adultos']):.4f}")
print(f"Delta Geo  (Original 80/20): {abs(rmse_80['USA/Can'] - rmse_80['Resto Mundo']):.4f} | Nuevo (50/50): {abs(rmse_50['USA/Can'] - rmse_50['Resto Mundo']):.4f}")
print(f"Delta Gusto(Original 80/20): {abs(rmse_80['Mainstream'] - rmse_80['Nicho']):.4f} | Nuevo (50/50): {abs(rmse_50['Mainstream'] - rmse_50['Nicho']):.4f}")

EXPERIMENTO 50/50: DELTAS ABSOLUTOS DE RMSE
Delta Edad (Original 80/20): 0.0839 | Nuevo (50/50): 0.0957
Delta Geo  (Original 80/20): 0.0123 | Nuevo (50/50): 0.0104
Delta Gusto(Original 80/20): 0.0608 | Nuevo (50/50): 0.0772


### Análisis y Resumen de Resultados

1. **Métricas Predictivas (RMSE/MAE):** Si bien se observan pequeñas diferencias en el RMSE entre grupos (por ejemplo, 1.5797 para Jóvenes vs 1.4958 para Adultos), las pruebas estadísticas de Mann-Whitney sobre el MAE indican que estas diferencias **no son estadísticamente significativas** (todos los p-values superan el umbral de 0.05). A nivel predictivo, el SVD no presenta un sesgo grave.

2. **Métricas de Ranking (Precision@5, Recall@5 y Cobertura):** Al simular un escenario de recomendación real (Top-5), el panorama cambia. El modelo favorece de manera drástica a los usuarios **Mainstream** frente a los de **Nicho**. Mientras que un usuario Mainstream tiene un P@5 de 0.1243 y R@5 de 0.1776, un usuario de Nicho apenas alcanza un P@5 de 0.0252 y un R@5 de 0.0352. Estas diferencias son altamente significativas a nivel estadístico (p-value muy cercano a cero). Además, los usuarios de Nicho reciben una diversidad de catálogo extremadamente pobre (cobertura del 1.62% frente a un 5.05%).

3. **Impacto de la reducción de datos (50/50):** Al reducir el tamaño del dataset, la disparidad de error tiende a ampliarse en los grupos críticos. La diferencia absoluta de RMSE entre usuarios Mainstream y Nicho creció de 0.0608 a 0.0772, y entre Jóvenes y Adultos de 0.0839 a 0.0957. Esto comprueba que la escasez de datos afecta de forma desproporcionada a ciertos perfiles, impactando negativamente en la equidad (Fairness) del sistema.

# Declaración de uso de herramientas de IA generativa

En el desarrollo de este trabajo práctico se utilizaron herramientas de Inteligencia Artificial generativa como asistencia técnica, bajo las siguientes condiciones:

- **Herramienta:** ChatGPT (OpenAI - Modelo GPT-4).
  - **Tareas:** Resolución de dudas sobre el uso de la librería Surprise para métricas de ranking, sintaxis para operaciones eficientes en Pandas y consultas sobre tests estadísticos no paramétricos.
  - **Prompts principales utilizados:** 
    1. *"¿Cuál es el test estadístico más adecuado en Python (scipy) para comparar los errores absolutos de predicción entre dos grupos de usuarios de tamaños distintos, asumiendo que los errores no siguen una distribución normal?"*
    2. *"En la librería Surprise, el método `test()` solo evalúa las interacciones reales. ¿Cómo puedo implementar de forma eficiente en Python un cálculo de Precision@5 y Recall@5 muestreando 100 ítems negativos (no interactuados) por cada usuario?"*
    3. *"Tengo un dataset de interacciones y un train/test split en Surprise. Escribí este código para clasificar a los usuarios en 'Nicho' o 'Mainstream', ¿puedes revisarlo para asegurarte de que no haya data leakage desde el testset hacia el cálculo de popularidad?"*

- **Herramienta:** GitHub Copilot.
  - **Tareas:** Autocompletado de código rutinario, formateo de strings en los `print()` para mejorar la visualización de los resultados de métricas y cierre de bucles.
  - **Prompts/Instrucciones:** No se utilizaron prompts directos, sino sugerencias de autocompletado en el entorno de desarrollo durante la redacción del script (ej. autocompletado en el diccionario de iteración de los grupos de evaluación).

*Se deja constancia de que todo el código generado o sugerido fue analizado, validado matemáticamente y adaptado manualmente para cumplir estrictamente con los requerimientos de la consigna de la materia.*